In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive
from math import log, exp
from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

# dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_20251111.csv')

In [3]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

# POINTS 

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_ts = league_df['TS_PCT'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_or = team_stats.at[player_team, 'OFF_RATING']
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']
    team_ts = team_stats.at[player_team, 'TS_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']

    # Opponent stats
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']

    df = s26
    try:
        player_df = df[df["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['PTS'].mean()
        else:
            lambda_base = player_df['PTS'].mean()
    except:
        lambda_base = player_df['PTS'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.93
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.04
    else:
        rest_factor = 0.98
    
    # Team offensive strength relative to league
    team_or_factor = cap_factor(team_or / league_avg_off_rtg)

    # Team assist-to-turnover ratio relative to league
    team_ast_ratio_factor = cap_factor(team_ast_ratio / league_avg_ast_ratio)
    
    # Team true shooting percentage relative to league
    team_ts_factor = cap_factor(team_ts / league_avg_ts)
    
    # Team offensive rebound percentage relative to league
    team_oreb_factor = cap_factor(team_oreb / league_avg_oreb)
    
    # Opponent defensive weakness relative to league (flip: lower def rating helps offense)
    opp_dr_factor = cap_factor(opp_dr / league_avg_def_rtg)
    
    # Pace adjustment relative to league
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (typical ~3% boost)
    home_factor = cap_factor(1.03 if home_flag else 0.97)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['PTS'].tail(7).mean()
    season_avg = player_df['PTS'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Usage rate adjustment (last 7 games vs season average)
    recent_usg_avg = player_df['USG_PCT'].tail(7).mean()
    season_usg_avg = player_df['USG_PCT'].mean()
    usg_factor = cap_factor(recent_usg_avg / season_usg_avg if season_usg_avg > 0 else 1.0)

    # Head-to-head adjustment (overall vs season average)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['PTS'].mean()
    else:
        h2h_avg = h2h['PTS'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_or_factor * 
                      opp_dr_factor * 
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      usg_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Andrew Wiggins,18.5,0.6,0.5,0.33,17.82,26.94,1.51,0.954,0.046,1.048
1,Cam Spencer,8.5,0.8,0.5,0.33,9.09,12.88,1.42,0.895,0.105,1.118
2,Tari Eason,11.5,0.6,0.5,0.33,11.44,16.35,1.43,0.889,0.111,1.125
3,Norman Powell,23.5,0.4,0.4,0.27,24.50,29.48,1.20,0.867,0.133,1.154
4,Russell Westbrook,12.5,0.6,0.5,0.33,15.00,16.83,1.12,0.856,0.144,1.168
5,Andre Drummond,8.5,0.4,0.2,0.13,6.89,11.39,1.65,0.800,0.200,1.249
6,Svi Mykhailiuk,8.5,0.6,0.6,0.40,9.30,11.13,1.20,0.780,0.220,1.282
7,Landry Shamet,7.5,0.6,0.5,0.33,7.00,9.71,1.39,0.753,0.247,1.328
8,Jalen Duren,19.5,0.6,0.6,0.40,19.36,22.83,1.18,0.751,0.249,1.331
9,Josh Hart,9.5,0.8,0.4,0.27,8.25,11.91,1.44,0.750,0.250,1.334


# ASSISTS

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']

    # Opponent stats
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['AST'].mean()
        else:
            lambda_base = player_df['AST'].mean()
    except:
        lambda_base = player_df['AST'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.93
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.04
    else:
        rest_factor = 0.98
    
    # Team assist culture (pass-heavy teams create more assists)
    team_ast_ratio_factor = cap_factor(team_ast_ratio / league_avg_ast_ratio)
    
    # Opponent defensive weakness (weak defense = easier passes)
    opp_dr_factor = cap_factor(opp_dr / league_avg_def_rtg)
    
    # Opponent turnover pressure (teams that force turnovers limit assists)
    opp_tov_factor = cap_factor(league_avg_tov / opp_tov)
    
    # Pace adjustment (more possessions = more assist opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (smaller effect for assists)
    home_factor = cap_factor(1.02 if home_flag else 0.98)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['AST'].tail(7).mean()
    season_avg = player_df['AST'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Usage rate adjustment (last 7 games vs season average)
    recent_usg_avg = player_df['USG_PCT'].tail(7).mean()
    season_usg_avg = player_df['USG_PCT'].mean()
    usg_factor = cap_factor(recent_usg_avg / season_usg_avg if season_usg_avg > 0 else 1.0)

    # Head-to-head adjustment (assists vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['AST'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['AST'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_ast_ratio_factor * 
                      opp_dr_factor * 
                      opp_tov_factor *
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      usg_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Nique Clifford,1.5,0.8,0.5,0.33,2.50,3.07,1.23,0.811,0.189,1.233
1,Jock Landale,1.5,0.6,0.5,0.33,1.64,2.82,1.72,0.772,0.228,1.296
2,Ja Morant,7.5,0.8,0.6,0.40,7.90,9.63,1.22,0.745,0.255,1.343
3,Cedric Coward,2.5,0.8,0.6,0.40,2.91,3.82,1.31,0.734,0.266,1.362
4,Isaiah Collier,4.5,0.2,0.1,0.07,6.28,5.97,0.95,0.710,0.290,1.408
5,Tari Eason,1.5,0.4,0.5,0.33,1.78,2.38,1.34,0.686,0.314,1.457
6,Josh Minott,1.5,0.4,0.4,0.27,1.40,1.92,1.37,0.572,0.428,1.747
7,Jerami Grant,2.5,0.8,0.5,0.33,2.40,2.95,1.23,0.565,0.435,1.769
8,Duncan Robinson,1.5,0.4,0.4,0.27,1.73,1.88,1.09,0.561,0.439,1.784
9,Christian Braun,2.5,0.6,0.5,0.33,3.11,2.81,0.90,0.533,0.467,1.877


# REBOUNDS

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_reb = team_stats.at[player_team, 'REB_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']
    team_dreb = team_stats.at[player_team, 'DREB_PCT']

    # Opponent stats
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_reb = team_stats.at[opp_team_id, 'REB_PCT']
    opp_oreb = team_stats.at[opp_team_id, 'OREB_PCT']
    opp_dreb = team_stats.at[opp_team_id, 'DREB_PCT']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['REB'].mean()
        else:
            lambda_base = player_df['REB'].mean()
    except:
        lambda_base = player_df['REB'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.95  # Slightly less penalty than scoring
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.03
    else:
        rest_factor = 0.99
    
    # Team rebounding culture (rebounding-focused teams)
    team_reb_factor = cap_factor(team_reb / league_avg_reb)
    
    # Opponent gives up offensive rebounds (weak DREB% = more offensive boards available)
    opp_dreb_factor = cap_factor(league_avg_dreb / opp_dreb)
    
    # Opponent gives up defensive rebounds (weak OREB% = more defensive boards available)
    opp_oreb_factor = cap_factor(league_avg_oreb / opp_oreb)
    
    # Pace adjustment (more possessions = more missed shots = more rebounds)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (minimal effect for rebounds)
    home_factor = cap_factor(1.01 if home_flag else 0.99)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['REB'].tail(7).mean()
    season_avg = player_df['REB'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Head-to-head adjustment (rebounds vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['REB'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['REB'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_reb_factor * 
                      opp_dreb_factor * 
                      opp_oreb_factor *
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Keyonte George,3.5,0.6,0.5,0.33,3.80,5.67,1.49,0.817,0.183,1.224
1,Jamal Shead,1.5,0.6,0.6,0.40,1.90,2.54,1.34,0.720,0.280,1.388
2,Jalen Duren,13.5,0.6,0.4,0.27,12.00,14.98,1.25,0.635,0.365,1.575
3,Moses Moody,2.5,0.8,0.6,0.40,2.89,3.25,1.12,0.630,0.370,1.588
4,Precious Achiuwa,4.5,0.2,0.1,0.07,5.56,5.38,0.97,0.624,0.376,1.602
5,Domantas Sabonis,12.5,0.8,0.5,0.33,14.00,13.55,0.97,0.596,0.404,1.677
6,Duncan Robinson,2.5,0.4,0.6,0.40,2.64,2.96,1.12,0.567,0.433,1.764
7,Jaylin Williams,4.5,0.4,0.5,0.33,4.45,5.03,1.13,0.565,0.435,1.770
8,Shaedon Sharpe,4.5,0.6,0.5,0.33,5.10,5.01,0.98,0.561,0.439,1.782
9,Jrue Holiday,4.5,0.6,0.8,0.53,5.50,4.99,0.91,0.558,0.442,1.792


# BLOCK

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']

    # Opponent stats
    opp_pace = team_stats.at[opp_team_id, 'PACE']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['BLK'].mean()
        else:
            lambda_base = player_df['BLK'].mean()
    except:
        lambda_base = player_df['BLK'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.97  # Back-to-back games reduce blocks slightly
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.03
    else:
        rest_factor = 0.99
    
    # Pace adjustment (more possessions = more shot attempts = more block opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (minimal effect for blocks)
    home_factor = cap_factor(1.00 if home_flag else 1.00)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['BLK'].tail(7).mean()
    season_avg = player_df['BLK'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Head-to-head adjustment (blocks vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['BLK'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['BLK'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
blocks_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_blocks.csv', index=False)
blocks_df.head(10)


,NAME,LINE,L-5,L-10,L-15,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Donovan Clingan,1.5,0.4,0.4,0.27,1.30,1.24,0.95,0.352,0.648,2.841
1,Chet Holmgren,1.5,0.4,0.2,0.13,1.29,1.12,0.87,0.307,0.693,3.262


# STEALS

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']

    # Opponent stats
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['STL'].mean()
        else:
            lambda_base = player_df['STL'].mean()
    except:
        lambda_base = player_df['STL'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.96  # Back-to-back games reduce steals (requires energy/focus)
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.03
    else:
        rest_factor = 0.99
    
    # Opponent turnover rate (higher TOV% = more steal opportunities)
    opp_tov_factor = cap_factor(opp_tov / league_avg_tov)
    
    # Pace adjustment (more possessions = more steal opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (minimal effect for steals)
    home_factor = cap_factor(1.00 if home_flag else 1.00)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['STL'].tail(7).mean()
    season_avg = player_df['STL'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Head-to-head adjustment (steals vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['STL'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['STL'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (opp_tov_factor *
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
steals_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
steals_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_steals.csv', index=False)
steals_df.head(10)

,NAME,LINE,L-5,L-10,L-15,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,OG Anunoby,1.5,0.6,0.6,0.40,2.22,2.01,0.91,0.597,0.403,1.674
1,Javonte Green,1.5,0.2,0.2,0.13,1.00,1.32,1.32,0.379,0.621,2.637
